In [5]:
!pip install kaggle wandb -Uq
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!rm -rf ~/.kaggle
!mkdir ~/.kaggle
!cp "/content/drive/MyDrive/Colab Notebooks/kaggle_API_credentials/kaggle.json" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [7]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -qn challenges-in-representation-learning-facial-expression-recognition-challenge.zip -d fer2013

challenges-in-representation-learning-facial-expression-recognition-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)


In [8]:
from google.colab import userdata
import wandb
wandb.login(key=userdata.get('WANDB_API_KEY'))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ngval22 (ngval22-s) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image
import itertools
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [10]:
df = pd.read_csv('./fer2013/train.csv')

train_val_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['emotion'])
train_df, val_df = train_test_split(train_val_df, test_size=0.111, random_state=42, stratify=train_val_df['emotion'])

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 22969, Val: 2869, Test: 2871


In [11]:
class_weights = compute_class_weight('balanced', classes=np.arange(7), y=train_df['emotion'].values)
class_weights[2] = 2.0
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print("Class weights:", class_weights)

Class weights: [1.026364   9.42898194 2.         0.56848332 0.84919403 1.29337237
 0.82589623]


In [12]:
class FERDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        pixels = np.array(row['pixels'].split(' '), dtype=np.uint8).reshape(48, 48)
        image = Image.fromarray(pixels)  # fixed: no mode='L', inferred from 2D array
        if self.transform:
            image = self.transform(image)
        label = int(row['emotion'])
        return image, label

In [13]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomCrop(48, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # new: adds lighting variation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5077], std=[0.2550])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5077], std=[0.2550])
])

In [14]:
class DeepCNN(nn.Module):
    """
    Architecture 3 — Deep CNN.
    Backward checking conclusion from arch2:
    - GAP(128) was too aggressive a bottleneck, destroyed spatial info
    - Dropout2d(0.25) on shallow feature maps caused training instability
    - Result: arch2 (44.4%) was WORSE than baseline (50.1%)

    Fixes applied:
    - Replace GAP with MaxPool + Flatten, keeping 512-dim classifier input
    - Remove Dropout2d from conv blocks entirely
    - Add a 3rd conv block (128->256 filters) for more capacity
    - Reduce classifier dropout to 0.3
    - Keep BatchNorm: it was working fine in arch2
    - Increase patience on scheduler to 7
    """
    def __init__(self, num_classes=7, dropout_rate=0.3):
        super(DeepCNN, self).__init__()

        self.features = nn.Sequential(
            # Block 1: 48x48 -> 24x24
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 2: 24x24 -> 12x12
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3: 12x12 -> 6x6
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),             # 256 * 6 * 6 = 9216
            nn.Linear(9216, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = DeepCNN(num_classes=7).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

DeepCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU()
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding

In [15]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

In [17]:
# Limited W&B sweep for low compute.
# Increase SWEEP_COUNT, SWEEP_EPOCHS, or FINAL_EPOCHS only when you have more GPU time.

PROJECT_NAME = "facial-expression-recognition"
SWEEP_COUNT = 4
SWEEP_EPOCHS = 20
FINAL_EPOCHS = 60

BATCH_SIZE = 64
NUM_WORKERS = 2

train_loader = DataLoader(FERDataset(train_df, transform=train_transform),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(FERDataset(val_df, transform=val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(FERDataset(test_df, transform=val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

sweep_config = {
    "method": "bayes",
    "metric": {"name": "best_val_acc", "goal": "maximize"},
    "parameters": {
        "learning_rate": {"values": [1e-3, 7e-4, 3e-4]},
        "dropout_rate": {"values": [0.3, 0.4, 0.5]},
        "weight_decay": {"values": [0.0, 1e-4]},
        "optimizer": {"values": ["Adam", "AdamW"]},
    }
}

best_sweep = {"val_acc": 0.0, "config": None, "checkpoint": None}

def make_optimizer(model, optimizer_name, learning_rate, weight_decay):
    if optimizer_name == "AdamW":
        return optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    return optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

def sweep_train():
    run = wandb.init(group="arch3-deep-sweep-limited")
    cfg = wandb.config

    sweep_model = DeepCNN(num_classes=7, dropout_rate=cfg.dropout_rate).to(device)
    sweep_criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    sweep_optimizer = make_optimizer(sweep_model, cfg.optimizer, cfg.learning_rate, cfg.weight_decay)

    best_val_acc = 0.0
    checkpoint_path = f"sweep_best_{run.id}.pth"

    for epoch in range(SWEEP_EPOCHS):
        train_loss, train_acc = train_one_epoch(sweep_model, train_loader, sweep_optimizer, sweep_criterion)
        val_loss, val_acc = evaluate(sweep_model, val_loader, sweep_criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(sweep_model.state_dict(), checkpoint_path)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "best_val_acc": best_val_acc,
        })

        print(f"Sweep {run.id} | Epoch {epoch+1:02d}/{SWEEP_EPOCHS} | "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    wandb.summary["best_val_acc"] = best_val_acc

    if best_val_acc > best_sweep["val_acc"]:
        best_sweep["val_acc"] = best_val_acc
        best_sweep["config"] = dict(cfg)
        best_sweep["checkpoint"] = checkpoint_path

    wandb.finish()

sweep_id = wandb.sweep(sweep_config, project=PROJECT_NAME)
wandb.agent(sweep_id, function=sweep_train, count=SWEEP_COUNT)

best_cfg = {
    "lr": best_sweep["config"]["learning_rate"],
    "dropout": best_sweep["config"]["dropout_rate"],
    "weight_decay": best_sweep["config"]["weight_decay"],
    "optimizer": best_sweep["config"]["optimizer"],
}

print(f"\nBest sweep config: {best_cfg} -> val_acc={best_sweep['val_acc']:.4f}")

# Log the best model from the sweep after the full search completes.
best_sweep_run = wandb.init(
    project=PROJECT_NAME,
    name="arch3-best-sweep-model",
    group="arch3-deep-sweep-limited",
    job_type="best_sweep_model",
    config={**best_cfg, "best_val_acc": best_sweep["val_acc"], "sweep_id": sweep_id}
)
artifact = wandb.Artifact("arch3-best-sweep-checkpoint", type="model")
artifact.add_file(best_sweep["checkpoint"])
best_sweep_run.log_artifact(artifact)
wandb.finish()

Create sweep with ID: owmhbmdl
Sweep URL: https://wandb.ai/ngval22-s/facial-expression-recognition/sweeps/owmhbmdl


wandb: Agent Starting Run: ickm550m with config:
wandb: 	dropout_rate: 0.4
wandb: 	learning_rate: 0.001
wandb: 	optimizer: Adam
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Sweep ickm550m | Epoch 01/20 | Train Acc: 0.1451 | Val Acc: 0.1426
Sweep ickm550m | Epoch 02/20 | Train Acc: 0.1489 | Val Acc: 0.1635
Sweep ickm550m | Epoch 03/20 | Train Acc: 0.1640 | Val Acc: 0.1558
Sweep ickm550m | Epoch 04/20 | Train Acc: 0.1680 | Val Acc: 0.1659
Sweep ickm550m | Epoch 05/20 | Train Acc: 0.1745 | Val Acc: 0.1785
Sweep ickm550m | Epoch 06/20 | Train Acc: 0.1789 | Val Acc: 0.1844
Sweep ickm550m | Epoch 07/20 | Train Acc: 0.2154 | Val Acc: 0.2684
Sweep ickm550m | Epoch 08/20 | Train Acc: 0.2726 | Val Acc: 0.3332
Sweep ickm550m | Epoch 09/20 | Train Acc: 0.2942 | Val Acc: 0.3381
Sweep ickm550m | Epoch 10/20 | Train Acc: 0.3326 | Val Acc: 0.3949
Sweep ickm550m | Epoch 11/20 | Train Acc: 0.3550 | Val Acc: 0.3747
Sweep ickm550m | Epoch 12/20 | Train Acc: 0.3707 | Val Acc: 0.4608
Sweep ickm550m | Epoch 13/20 | Train Acc: 0.3948 | Val Acc: 0.3754
Sweep ickm550m | Epoch 14/20 | Train Acc: 0.4254 | Val Acc: 0.4549
Sweep ickm550m | Epoch 15/20 | Train Acc: 0.4269 | Val Acc: 0.

best_val_acc,▁▁▁▁▂▂▃▄▅▆▆▇▇▇▇▇▇███
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▁▁▁▂▂▂▄▄▅▅▆▆▇▇▇▇▇██
train_loss,█▇▇▆▆▆▆▅▅▄▄▄▃▃▂▂▂▁▁▁
val_acc,▁▁▁▁▂▂▃▄▅▆▅▇▅▇▆▇▇█▆█
val_loss,█▇▇▆▆▆▆▅▄▄▃▃▃▂▂▁▁▁▁▁
best_val_acc,0.53224
epoch,20
train_acc,0.48988
train_loss,1.27694
val_acc,0.53224


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: 7cos034j with config:
wandb: 	dropout_rate: 0.4
wandb: 	learning_rate: 0.0003
wandb: 	optimizer: Adam
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


...

best_val_acc,▁▄▄▅▆▆▆▇▇▇██████████
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▁▃▄▅▆▆▆▇▇▇▇▇▇▇█████
train_loss,█▇▆▆▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁
val_acc,▁▄▄▅▆▆▆▇▇▇█▇▇▇▇█▇▇▇█
val_loss,█▇▆▅▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
best_val_acc,0.5176
epoch,20
train_acc,0.48761
train_loss,1.23304
val_acc,0.5176


wandb: Agent Starting Run: v059cg0v with config:
wandb: 	dropout_rate: 0.5
wandb: 	learning_rate: 0.001
wandb: 	optimizer: Adam
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Sweep v059cg0v | Epoch 01/20 | Train Acc: 0.1415 | Val Acc: 0.1426
Sweep v059cg0v | Epoch 02/20 | Train Acc: 0.1433 | Val Acc: 0.1426
Sweep v059cg0v | Epoch 03/20 | Train Acc: 0.1438 | Val Acc: 0.1426
Sweep v059cg0v | Epoch 04/20 | Train Acc: 0.1426 | Val Acc: 0.1426
Sweep v059cg0v | Epoch 05/20 | Train Acc: 0.1428 | Val Acc: 0.1426
Sweep v059cg0v | Epoch 06/20 | Train Acc: 0.1426 | Val Acc: 0.1433
Sweep v059cg0v | Epoch 07/20 | Train Acc: 0.1432 | Val Acc: 0.1426
Sweep v059cg0v | Epoch 08/20 | Train Acc: 0.1432 | Val Acc: 0.1426
Sweep v059cg0v | Epoch 09/20 | Train Acc: 0.1425 | Val Acc: 0.1426
Sweep v059cg0v | Epoch 10/20 | Train Acc: 0.1426 | Val Acc: 0.1464
Sweep v059cg0v | Epoch 11/20 | Train Acc: 0.1421 | Val Acc: 0.1426
Sweep v059cg0v | Epoch 12/20 | Train Acc: 0.1447 | Val Acc: 0.1433
Sweep v059cg0v | Epoch 13/20 | Train Acc: 0.1553 | Val Acc: 0.1673
Sweep v059cg0v | Epoch 14/20 | Train Acc: 0.1748 | Val Acc: 0.1987
Sweep v059cg0v | Epoch 15/20 | Train Acc: 0.2062 | Val Acc: 0.

best_val_acc,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▅▅▆▇▇█
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▅▆▇██
train_loss,█▇▇▇▇▇▇▇▇▇▇▇▇▆▅▄▃▂▂▁
val_acc,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▅▅▆▇▇█
val_loss,███████████▇▇▆▅▄▂▂▁▁
best_val_acc,0.50889
epoch,20
train_acc,0.41569
train_loss,1.43217
val_acc,0.50889



Best sweep config: {'lr': 0.001, 'dropout': 0.4, 'weight_decay': 0.0001, 'optimizer': 'Adam'} -> val_acc=0.5322


In [18]:
config = {
    "architecture": "DeepCNN",
    "epochs": FINAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": best_cfg['lr'],
    "dropout_rate": best_cfg['dropout'],
    "weight_decay": best_cfg['weight_decay'],
    "optimizer": best_cfg['optimizer'],
    "scheduler": "CosineAnnealingLR",
    "loss": "CrossEntropyLoss with class weights",
    "num_conv_blocks": 3,
    "num_conv_layers": 6,
    "batch_norm": True,
    "global_avg_pooling": False,
    "phase": "limited_full_run_after_sweep",
    "notes": "Limited final training from the best W&B sweep config to save compute."
}

run = wandb.init(
    project=PROJECT_NAME,
    name="arch3-deep-cnn-sweep-best-limited",
    group="arch3-deep-fullrun-limited",
    config=config
)

model = DeepCNN(num_classes=7, dropout_rate=best_cfg['dropout']).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = make_optimizer(model, best_cfg['optimizer'], best_cfg['lr'], best_cfg['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINAL_EPOCHS, eta_min=1e-5)

best_val_acc = 0.0
best_model_path = 'best_deep_cnn_sweep_limited.pth'

for epoch in range(config['epochs']):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = evaluate(model, val_loader, criterion)
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)

    wandb.log({
        "epoch":      epoch + 1,
        "train_loss": train_loss,
        "train_acc":  train_acc,
        "val_loss":   val_loss,
        "val_acc":    val_acc,
        "best_val_acc": best_val_acc,
        "lr":         optimizer.param_groups[0]['lr'],
    })

    print(f"Epoch {epoch+1:02d}/{config['epochs']} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print(f"\nBest Val Accuracy: {best_val_acc:.4f}")

Epoch 01/60 | Train Loss: 1.9545 Acc: 0.1416 | Val Loss: 1.9063 Acc: 0.1426
Epoch 02/60 | Train Loss: 1.9121 Acc: 0.1427 | Val Loss: 1.9064 Acc: 0.1426
Epoch 03/60 | Train Loss: 1.9060 Acc: 0.1433 | Val Loss: 1.9254 Acc: 0.1426
Epoch 04/60 | Train Loss: 1.9066 Acc: 0.1427 | Val Loss: 1.9066 Acc: 0.1426
Epoch 05/60 | Train Loss: 1.9070 Acc: 0.1431 | Val Loss: 1.9064 Acc: 0.1426
Epoch 06/60 | Train Loss: 1.9063 Acc: 0.1427 | Val Loss: 1.9075 Acc: 0.1426
Epoch 07/60 | Train Loss: 1.9076 Acc: 0.1427 | Val Loss: 1.9064 Acc: 0.1426
Epoch 08/60 | Train Loss: 1.9070 Acc: 0.1427 | Val Loss: 1.9064 Acc: 0.1426
Epoch 09/60 | Train Loss: 1.9079 Acc: 0.1427 | Val Loss: 1.9048 Acc: 0.1426
Epoch 10/60 | Train Loss: 1.9107 Acc: 0.1425 | Val Loss: 1.9040 Acc: 0.1426
Epoch 11/60 | Train Loss: 1.9009 Acc: 0.1453 | Val Loss: 1.8890 Acc: 0.1450
Epoch 12/60 | Train Loss: 1.8896 Acc: 0.1458 | Val Loss: 1.8660 Acc: 0.1436
Epoch 13/60 | Train Loss: 1.8550 Acc: 0.1572 | Val Loss: 1.8103 Acc: 0.1690
Epoch 14/60 

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

wandb.log({"test_loss": test_loss, "test_acc": test_acc, "best_val_acc": best_val_acc})

In [ ]:
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
class_correct = [0] * 7
class_total   = [0] * 7

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        for label, pred in zip(labels, predicted):
            class_correct[label] += (pred == label).item()
            class_total[label]   += 1

per_class = {}
print(f"{'Class':<12} {'Correct':<10} {'Total':<10} {'Accuracy'}")
print("-" * 42)
for i, name in enumerate(emotion_labels):
    acc = class_correct[i] / class_total[i] if class_total[i] > 0 else 0
    per_class[f"test_acc_{name.lower()}"] = acc
    print(f"{name:<12} {class_correct[i]:<10} {class_total[i]:<10} {acc:.4f}")

wandb.log(per_class)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=emotion_labels,
            yticklabels=emotion_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Deep CNN')
plt.tight_layout()
wandb.log({"confusion_matrix": wandb.Image(fig)})
plt.show()

In [ ]:
wandb.summary['best_val_acc'] = best_val_acc
wandb.summary['test_acc']     = test_acc

# Log the best checkpoint from the limited final training run.
artifact = wandb.Artifact("arch3-best-final-checkpoint", type="model")
artifact.add_file(best_model_path)
wandb.log_artifact(artifact)

wandb.finish()